### ***LOADING THE WINDOWED AUDIO***

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

NOTEBOOK_DIR:Path = Path.cwd()
REPOSITORY_ROOT:Path = NOTEBOOK_DIR
AUDIO_DATA_DIR_WINDOWED:Path = REPOSITORY_ROOT / "preprocessing-02-windowing.npy"
AUDIO_DATA_DIR_FFT:Path = REPOSITORY_ROOT / "preprocessing-03-fft.npy"

# only used for the plot's frequency axis. Set this to the rate you used when framing.
SAMPLE_RATE:int = 16_000

In [ ]:
LOADED_AUDIO_FRAMES:np.ndarray = np.load(AUDIO_DATA_DIR_WINDOWED.resolve(), allow_pickle=True)
print(f"Loaded {len(LOADED_AUDIO_FRAMES)} frames, shape {LOADED_AUDIO_FRAMES.shape}")
print("Columns: source_file, frame_number, samples")
print(f"Samples per frame: {len(LOADED_AUDIO_FRAMES[0][2])}")

### ***COMPUTING THE FFT***

Audio is real-valued, so `np.fft.rfft` is used. It returns only the non-negative frequencies (`N//2 + 1` bins), because the other half is a mirror image.

Each frame produces:
- **complex spectrum** : the raw FFT output (kept, in case phase is needed later)
- **magnitude spectrum** : `abs(fft)`
- **power spectrum** : `(1/N) * magnitude**2`, the usual input for mel filterbanks

In [ ]:
def process_frame_row(row) -> list:
    source_file, frame_number, samples = row
    spectrum:np.ndarray  = np.fft.rfft(samples)
    magnitude:np.ndarray = np.abs(spectrum)
    power:np.ndarray     = (magnitude ** 2) / (len(samples))
    return [source_file, frame_number, spectrum, magnitude.astype(np.float32), power.astype(np.float32)]


def build_fft_array(frames:np.ndarray) -> np.ndarray:
    out = np.empty((len(frames), 5), dtype=object)

    test_row = frames[0]
    source_file, frame_number, samples = test_row
    print(f"source_file = {type(source_file)}")
    print(f"frame_number = {type(frame_number)}")
    print(f"samples = {type(samples)}")
    for i, row in enumerate(frames):
        out[i] = process_frame_row(row)
    return out


FFT_FRAMES:np.ndarray = build_fft_array(LOADED_AUDIO_FRAMES)
print(f"Computed FFT for {len(FFT_FRAMES)} frames, shape {FFT_FRAMES.shape}")
print(f"Bins per frame: {len(FFT_FRAMES[0][3])}")

### ***INSPECTING ONE FRAME***

In [ ]:
n = 25
print("DIR", f"\t{FFT_FRAMES[n][0]=}", sep="\n")
print("="*50, "FRAME#", f"\t{FFT_FRAMES[n][1]=}", sep="\n")
print("="*50, "COMPLEX_SPECTRUM", f"\t{FFT_FRAMES[n][2]=}", sep="\n")
print("="*50, "MAGNITUDE", f"\t{FFT_FRAMES[n][3]=}", sep="\n")
print("="*50, "POWER", f"\t{FFT_FRAMES[n][4]=}", sep="\n")

In [ ]:
n = 25
samples   = LOADED_AUDIO_FRAMES[n][2]
magnitude = FFT_FRAMES[n][3]
freqs     = np.fft.rfftfreq(len(samples), d=1.0 / SAMPLE_RATE)

fig, (ax_time, ax_freq) = plt.subplots(2, 1, figsize=(10, 6))

ax_time.plot(samples)
ax_time.set_title(f"Windowed frame #{FFT_FRAMES[n][1]} (time domain)")
ax_time.set_xlabel("Sample")
ax_time.set_ylabel("Amplitude")

ax_freq.plot(freqs, magnitude)
ax_freq.set_title("FFT magnitude spectrum")
ax_freq.set_xlabel("Frequency (Hz)")
ax_freq.set_ylabel("Magnitude")

plt.tight_layout()
plt.show()

### ***SAVING THE FFT FILES***

In [ ]:
AUDIO_DATA_DIR_FFT.parent.mkdir(parents=True, exist_ok=True)
np.save(AUDIO_DATA_DIR_FFT.resolve(), FFT_FRAMES, allow_pickle=True)
# note: pickle can execute code on load, so only load files you created yourself.

print(f"Saved {len(FFT_FRAMES)} frames with shape {FFT_FRAMES.shape} to {AUDIO_DATA_DIR_FFT.resolve()}")
print("Columns: source_file, frame_number, complex_spectrum, magnitude, power")

### ***VERIFYING THE SAVED FFT FRAMES***

In [ ]:
LOADED_FFT_FRAMES:np.ndarray = np.load(AUDIO_DATA_DIR_FFT.resolve(), allow_pickle=True)
EXPECTED_BINS:int = len(LOADED_AUDIO_FRAMES[0][2]) // 2 + 1

assert LOADED_FFT_FRAMES.shape == FFT_FRAMES.shape, (
    f"Shape changed: {FFT_FRAMES.shape} -> {LOADED_FFT_FRAMES.shape}"
)
assert LOADED_FFT_FRAMES.shape[0] == LOADED_AUDIO_FRAMES.shape[0], (
    "Frame count differs from the windowed input."
)
assert LOADED_FFT_FRAMES.dtype == FFT_FRAMES.dtype, (
    f"Data type changed: {FFT_FRAMES.dtype} -> {LOADED_FFT_FRAMES.dtype}"
)
assert np.array_equal(LOADED_FFT_FRAMES[:, 0], LOADED_AUDIO_FRAMES[:, 0]), (
    "One or more source-file paths differ from the windowed input."
)
assert np.array_equal(LOADED_FFT_FRAMES[:, 1], LOADED_AUDIO_FRAMES[:, 1]), (
    "One or more frame numbers differ from the windowed input."
)
for source_file in np.unique(LOADED_FFT_FRAMES[:, 0]):
    frame_numbers = LOADED_FFT_FRAMES[
        LOADED_FFT_FRAMES[:, 0] == source_file, 1
    ].astype(np.int64)
    assert np.array_equal(
        frame_numbers, np.arange(1, len(frame_numbers) + 1)
    ), f"Frame sequence is not continuous for {source_file}"

for fft_row, window_row in zip(LOADED_FFT_FRAMES, LOADED_AUDIO_FRAMES, strict=True):
    spectrum, magnitude, power = fft_row[2], fft_row[3], fft_row[4]
    samples = window_row[2]

    # shape: rfft gives N//2 + 1 bins
    assert len(spectrum) == len(magnitude) == len(power) == EXPECTED_BINS, (
        f"Wrong bin count in frame {fft_row[1]} of {fft_row[0]}"
    )
    # values: no NaN or inf, power is never negative
    assert np.isfinite(magnitude).all() and np.isfinite(power).all(), (
        f"NaN or inf in frame {fft_row[1]} of {fft_row[0]}"
    )
    assert (power >= 0).all(), f"Negative power in frame {fft_row[1]} of {fft_row[0]}"
    # recompute from the windowed samples and compare
    expected_spectrum = np.fft.rfft(samples)
    assert np.allclose(spectrum, expected_spectrum), (
        f"Spectrum differs from a fresh FFT in frame {fft_row[1]} of {fft_row[0]}"
    )
    assert np.allclose(magnitude, np.abs(expected_spectrum), rtol=1e-4, atol=1e-8), (
        f"Magnitude differs in frame {fft_row[1]} of {fft_row[0]}"
    )
    assert np.allclose(power, np.abs(expected_spectrum) ** 2 / len(samples), rtol=1e-4, atol=1e-12), (
        f"Power differs in frame {fft_row[1]} of {fft_row[0]}"
    )

print(
    f"Integrity verified: {len(LOADED_FFT_FRAMES)} saved FFT frames match "
    "the windowed frames they came from."
)